In [12]:
# =========================
# CONFIG (edit these)
# =========================

HANDLE = "https://www.youtube.com/@sdpnoticias"

OUT_PARQUET = "../../data/02-conferences/auxiliar/milenio_all_videos.parquet"
WRITE_CSV_TOO = True
MAKE_PLOT = True
AUTO_INSTALL = True

In [13]:
# =========================
# Dependencies
# =========================

import sys, subprocess

def ensure_package(pkg: str):
    try:
        __import__(pkg)
    except Exception:
        if not AUTO_INSTALL:
            raise
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", pkg])

ensure_package("pandas")
ensure_package("pyarrow")
ensure_package("googleapiclient")
ensure_package("dotenv")
ensure_package("matplotlib")

print("✅ Dependencies ready")

✅ Dependencies ready


In [14]:
# =========================
# Imports
# =========================

import os
import re
import json
import time
from datetime import datetime, timezone
from typing import Dict, List, Optional, Any

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

In [15]:
# =========================
# Helpers
# =========================

def iso_to_dt(iso_str: Optional[str]) -> Optional[pd.Timestamp]:
    if not iso_str:
        return None
    try:
        return pd.to_datetime(iso_str, utc=True)
    except Exception:
        return None

def chunked(xs: List[str], n: int) -> List[List[str]]:
    return [xs[i:i+n] for i in range(0, len(xs), n)]

def safe_json(x: Any) -> Optional[str]:
    if x is None:
        return None
    try:
        return json.dumps(x, ensure_ascii=False, sort_keys=True)
    except Exception:
        return str(x)

def parse_iso8601_duration_to_seconds(duration: Optional[str]) -> Optional[int]:
    if not duration or not isinstance(duration, str):
        return None
    m = re.fullmatch(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", duration)
    if not m:
        return None
    h = int(m.group(1) or 0)
    mm = int(m.group(2) or 0)
    s = int(m.group(3) or 0)
    return h * 3600 + mm * 60 + s

def call_with_retries(fn, max_retries: int = 6, base_sleep: float = 1.0):
    for attempt in range(max_retries):
        try:
            return fn()
        except HttpError as e:
            status = getattr(e.resp, "status", None)
            msg = str(e).lower()
            if status in (429, 500, 503) or (status == 403 and "quota" in msg):
                time.sleep(base_sleep * (2 ** attempt))
                continue
            raise
    raise RuntimeError("Max retries exceeded.")

def extract_handle(handle_input: str) -> str:
    """Extract bare handle (no @, no URL) from various input formats."""
    m = re.search(r"@([A-Za-z0-9._-]+)", handle_input)
    return m.group(1) if m else handle_input.strip().lstrip("@")

print("✅ Helpers loaded")

✅ Helpers loaded


In [16]:
# =========================
# API extractors
# =========================

def resolve_handle_to_uploads_playlist(youtube, handle: str) -> tuple:
    """Returns (channel_id, uploads_playlist_id) for a channel handle."""
    resp = call_with_retries(lambda: youtube.channels().list(
        part="id,snippet,contentDetails,statistics",
        forHandle=handle,
    ).execute())
    items = resp.get("items", [])
    if not items:
        raise RuntimeError(f"No channel found for handle: @{handle}")
    ch = items[0]
    channel_id = ch["id"]
    uploads_playlist_id = ch["contentDetails"]["relatedPlaylists"]["uploads"]
    stats = ch.get("statistics", {})
    print(f"Channel: {ch['snippet']['title']}  (id={channel_id})")
    print(f"Subscribers: {stats.get('subscriberCount', 'hidden')}  |  Videos: {stats.get('videoCount', '?')}")
    return channel_id, uploads_playlist_id


def fetch_playlist_items(youtube, playlist_id: str) -> Dict[str, Dict[str, Any]]:
    """Returns map: videoId -> playlist metadata (fetches all pages)."""
    out: Dict[str, Dict[str, Any]] = {}
    page_token = None
    page = 0

    while True:
        resp = call_with_retries(lambda: youtube.playlistItems().list(
            part="snippet,contentDetails",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=page_token,
        ).execute())

        for it in resp.get("items", []):
            snip = it.get("snippet", {}) or {}
            cd = it.get("contentDetails", {}) or {}
            vid = cd.get("videoId") or ((snip.get("resourceId") or {}).get("videoId"))
            if not vid:
                continue
            out[vid] = {
                "playlistItemId": it.get("id"),
                "playlist_position": snip.get("position"),
                "playlist_addedAt": snip.get("publishedAt"),
                "playlist_itemTitle": snip.get("title"),
                "playlist_videoPublishedAt": cd.get("videoPublishedAt"),
            }

        page_token = resp.get("nextPageToken")
        page += 1
        if page % 10 == 0:
            print(f"  ...fetched {len(out):,} video IDs so far")
        if not page_token:
            break

    print(f"✅ Total video IDs in uploads playlist: {len(out):,}")
    return out


def fetch_videos_details(youtube, video_ids: List[str]) -> List[Dict[str, Any]]:
    """Fetch per-video public fields in batches of 50."""
    parts = "snippet,statistics,contentDetails,status,liveStreamingDetails,topicDetails,recordingDetails"
    rows: List[Dict[str, Any]] = []

    for i, chunk in enumerate(chunked(video_ids, 50)):
        resp = call_with_retries(lambda: youtube.videos().list(
            part=parts,
            id=",".join(chunk),
            maxResults=50,
        ).execute())

        for v in resp.get("items", []):
            snip = v.get("snippet", {}) or {}
            stats = v.get("statistics", {}) or {}
            cd = v.get("contentDetails", {}) or {}
            st = v.get("status", {}) or {}
            live = v.get("liveStreamingDetails", {}) or {}
            topic = v.get("topicDetails", {}) or {}
            rec = v.get("recordingDetails", {}) or {}

            rows.append({
                "videoId": v.get("id"),
                "title": snip.get("title"),
                "description": snip.get("description"),
                "publishedAt": snip.get("publishedAt"),
                "channelId": snip.get("channelId"),
                "channelTitle": snip.get("channelTitle"),
                "categoryId": snip.get("categoryId"),
                "defaultLanguage": snip.get("defaultLanguage"),
                "defaultAudioLanguage": snip.get("defaultAudioLanguage"),
                "tags_json": safe_json(snip.get("tags")),
                "viewCount": int(stats["viewCount"]) if stats.get("viewCount") is not None else None,
                "likeCount": int(stats["likeCount"]) if stats.get("likeCount") is not None else None,
                "commentCount": int(stats["commentCount"]) if stats.get("commentCount") is not None else None,
                "favoriteCount": int(stats["favoriteCount"]) if stats.get("favoriteCount") is not None else None,
                "duration_iso": cd.get("duration"),
                "duration_seconds": parse_iso8601_duration_to_seconds(cd.get("duration")),
                "dimension": cd.get("dimension"),
                "definition": cd.get("definition"),
                "caption": cd.get("caption"),
                "licensedContent": cd.get("licensedContent"),
                "projection": cd.get("projection"),
                "regionRestriction_json": safe_json(cd.get("regionRestriction")),
                "uploadStatus": st.get("uploadStatus"),
                "privacyStatus": st.get("privacyStatus"),
                "license": st.get("license"),
                "embeddable": st.get("embeddable"),
                "madeForKids": st.get("madeForKids"),
                "live_actualStartTime": live.get("actualStartTime"),
                "live_actualEndTime": live.get("actualEndTime"),
                "live_scheduledStartTime": live.get("scheduledStartTime"),
                "live_concurrentViewers": int(live["concurrentViewers"]) if live.get("concurrentViewers") else None,
                "topicDetails_json": safe_json(topic if topic else None),
                "recordingDate": rec.get("recordingDate"),
            })

        if (i + 1) % 20 == 0:
            print(f"  ...fetched details for {len(rows):,} videos")

    print(f"✅ Video details fetched: {len(rows):,}")
    return rows

print("✅ API extractors loaded")

✅ API extractors loaded


In [17]:
# =========================
# Channel overview (run this first)
# =========================

load_dotenv()
API_KEY = "REDACTED_YOUTUBE_API_KEY"

youtube = build("youtube", "v3", developerKey=API_KEY)
handle = extract_handle(HANDLE)

resp = call_with_retries(lambda: youtube.channels().list(
    part="id,snippet,statistics,contentDetails,brandingSettings,topicDetails",
    forHandle=handle,
).execute())

ch = resp["items"][0]
snip  = ch.get("snippet", {})
stats = ch.get("statistics", {})
brand = ch.get("brandingSettings", {}).get("channel", {})
topic = ch.get("topicDetails", {})
cd    = ch.get("contentDetails", {}).get("relatedPlaylists", {})

print("=" * 50)
print(f"  Channel:      {snip.get('title')}")
print(f"  Handle:       {snip.get('customUrl')}")
print(f"  Country:      {snip.get('country')}")
print(f"  Created:      {snip.get('publishedAt')}")
print(f"  Language:     {snip.get('defaultLanguage')}")
print("=" * 50)
print(f"  Videos:       {int(stats.get('videoCount', 0)):,}")
print(f"  Subscribers:  {'hidden' if stats.get('hiddenSubscriberCount') else int(stats.get('subscriberCount', 0)):,}")
print(f"  Total views:  {int(stats.get('viewCount', 0)):,}")
print("=" * 50)
print(f"  Uploads PL:   {cd.get('uploads')}")
print(f"  Keywords:     {brand.get('keywords', '')[:80]}")
print(f"  Topics:       {topic.get('topicCategories', [])}")
print("=" * 50)
print(f"\nDescription:\n{snip.get('description', '')[:300]}")

  Channel:      SDPNoticias
  Handle:       @sdpnoticias
  Country:      MX
  Created:      2011-12-06T20:14:15Z
  Language:     None
  Videos:       19,284
  Subscribers:  993,000
  Total views:  597,381,328
  Uploads PL:   UUfV4ia-X7S7NCP4gmlBP1iA
  Keywords:     sdpnoticias sdp "sdp noticias" "estefania veloz" "poncho gutierrez" amlo "claudi
  Topics:       ['https://en.wikipedia.org/wiki/Politics', 'https://en.wikipedia.org/wiki/Society']

Description:
Bienvenido al canal oficial de SDPnoticias.com

Noticias de actualidad, temas relevantes en redes sociales, espectáculos, política, deportes, cultura, nacional e internacional.





In [18]:
# =========================
# RUN — fetch all videos from handle
# =========================

load_dotenv()
API_KEY = "REDACTED_YOUTUBE_API_KEY"

youtube = build("youtube", "v3", developerKey=API_KEY)
handle = extract_handle(HANDLE)

# 1) Resolve handle → channel → uploads playlist
channel_id, uploads_playlist_id = resolve_handle_to_uploads_playlist(youtube, handle)
print(f"Uploads playlist: {uploads_playlist_id}\n")

# 2) Fetch all video IDs (cheap — only playlist metadata)
playlist_meta = fetch_playlist_items(youtube, uploads_playlist_id)

# 3) Filter to videos published on or after 2018-01-01 before fetching details
cutoff = pd.Timestamp("2018-01-01", tz="UTC")
playlist_meta = {
    vid: meta for vid, meta in playlist_meta.items()
    if iso_to_dt(meta.get("playlist_videoPublishedAt") or meta.get("playlist_addedAt")) >= cutoff
}
video_ids = list(playlist_meta.keys())
print(f"Videos after Jan 2018: {len(video_ids):,}")

# 4) Fetch full details only for filtered videos
video_rows = fetch_videos_details(youtube, video_ids)
df = pd.DataFrame(video_rows)

# 5) Merge playlist metadata
pm = (pd.DataFrame.from_dict(playlist_meta, orient="index")
      .reset_index()
      .rename(columns={"index": "videoId"}))
df = df.merge(pm, on="videoId", how="left")

# 6) Normalize dates
for col in ["publishedAt", "playlist_addedAt", "live_actualStartTime", "live_actualEndTime"]:
    df[f"{col}_dt"] = df[col].apply(iso_to_dt)

df["collectedAt_utc"] = pd.Timestamp(datetime.now(timezone.utc))
df["channel_id"] = channel_id

# 7) Sort chronologically
df = df.sort_values("publishedAt_dt").reset_index(drop=True)

# 8) Save
out_dir = os.path.dirname(OUT_PARQUET) or "."
os.makedirs(out_dir, exist_ok=True)
df.to_parquet(OUT_PARQUET, index=False)
if WRITE_CSV_TOO:
    csv_path = re.sub(r"\.parquet$", "", OUT_PARQUET) + ".csv"
    df.to_csv(csv_path, index=False)
    print(f"CSV: {csv_path}")

print(f"\n✅ Saved: {OUT_PARQUET}")
print(f"Rows: {len(df):,}")
df[["publishedAt_dt", "viewCount", "likeCount", "commentCount", "duration_seconds", "title"]].tail(10)

Channel: SDPNoticias  (id=UCfV4ia-X7S7NCP4gmlBP1iA)
Subscribers: 993000  |  Videos: 19284
Uploads playlist: UUfV4ia-X7S7NCP4gmlBP1iA

  ...fetched 500 video IDs so far
  ...fetched 1,000 video IDs so far
  ...fetched 1,500 video IDs so far
  ...fetched 2,000 video IDs so far
  ...fetched 2,500 video IDs so far
  ...fetched 3,000 video IDs so far
  ...fetched 3,500 video IDs so far
  ...fetched 4,000 video IDs so far
  ...fetched 4,500 video IDs so far
  ...fetched 5,000 video IDs so far
  ...fetched 5,500 video IDs so far
  ...fetched 6,000 video IDs so far
  ...fetched 6,500 video IDs so far
  ...fetched 7,000 video IDs so far
  ...fetched 7,500 video IDs so far
  ...fetched 8,000 video IDs so far
  ...fetched 8,500 video IDs so far
  ...fetched 9,000 video IDs so far
  ...fetched 9,500 video IDs so far
  ...fetched 10,000 video IDs so far
  ...fetched 10,500 video IDs so far
  ...fetched 11,000 video IDs so far
  ...fetched 11,500 video IDs so far
  ...fetched 12,000 video IDs so far

,publishedAt_dt,viewCount,likeCount,commentCount,duration_seconds,title
17875,2026-03-06 20:40:52+00:00,3266,216,7,44.0,🚨 Luisa Alcalde desmiente rumores sobre su sal...
17876,2026-03-06 20:50:58+00:00,0,3,0,NaN,"""Alito"" busca ALIANZA con PAN y MC | 16 años, ..."
17877,2026-03-06 21:14:36+00:00,7365,204,26,68.0,"🚨La tumba de ‘El Mencho’ en Zapopan, está vigi..."
17878,2026-03-06 21:47:31+00:00,5127,467,72,44.0,💰 Revelan patrimonio de 22 millones de diputad...
17879,2026-03-06 22:04:55+00:00,550,83,2,50.0,#SDPMañana ☀️ con Liz Vilchis | 🎨🕊️ Adiós al m...
17880,2026-03-06 22:10:10+00:00,4699,569,13,76.0,🚨 SCJN cierra la puerta a los últimos amparos ...
17881,2026-03-06 22:15:30+00:00,801,85,0,78.0,#SDPMañana ☀️ con Liz Vilchis | Pablo Lemus se...
17882,2026-03-06 22:23:39+00:00,4724,245,17,91.0,🚨 Jorge Álvarez Máynez descarta alianza con el...
17883,2026-03-06 22:50:51+00:00,3665,418,6,2757.0,La FUNADA de Saskia Niño de Rivera | Un LIBRO ...
17884,2026-03-06 23:15:58+00:00,145,11,0,67.0,🎤 Cancelan conciertos de Pink en México y fans...


In [21]:
df.to_parquet('sdpnoticias.parquet', index=False)

In [26]:
df[df['publishedAt']<'2022']

,videoId,title,description,publishedAt,channelId,channelTitle,categoryId,defaultLanguage,defaultAudioLanguage,tags_json,...,playlist_position,playlist_addedAt,playlist_itemTitle,playlist_videoPublishedAt,publishedAt_dt,playlist_addedAt_dt,live_actualStartTime_dt,live_actualEndTime_dt,collectedAt_utc,channel_id


In [ ]:
# =========================
# Optional: views over time plot
# =========================

if MAKE_PLOT:
    dff = df.dropna(subset=["publishedAt_dt", "viewCount"]).copy()
    plt.figure(figsize=(14, 4))
    plt.plot(dff["publishedAt_dt"], dff["viewCount"], alpha=0.7)
    plt.xlabel("Published (UTC)")
    plt.ylabel("Views (snapshot)")
    plt.title(f"@{handle} — Views by publication date ({len(dff):,} videos)")
    plt.tight_layout()
    plt.show()